# LA 02 — Causal second-touch candidates and flow features

**Status:** exploratory feature extraction, not a trading strategy backtest. The data-readiness audit (LA 01 v0.1.2) has passed for BTC/ETH historical Binance USD-M perpetual candles and sampled aggregated trades. This notebook constructs **price-defined** candidate events, then attaches delta and traded-volume-at-price observations available by the *end of the second rejection candle*. It does **not** calculate post-entry returns, stops, targets, profitability, or claim causality of hidden orders.

Price-zone anchors are proxies: the **previous completed UTC day high/low** and the latest **fully confirmed five-minute pivot**. They are **not** yet volume-profile-derived discretionary levels. Research parameters are declared in code and saved to `02_feature_contract.json`; they have **not** been optimized or formally frozen for trading evaluation. Trade-volume-at-level uses **1bp price bins**, so it is approximate.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / "src" / "data_engineering").exists():
    if REPO == REPO.parent:
        raise RuntimeError("Launch Jupyter from repository or a subdirectory")
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.data_engineering.modeling.liquidity_absorption_stage_a import binance_root
from src.data_engineering.modeling.liquidity_absorption_stage_a02 import Config, LA_A02_VERSION, run_a02
assert LA_A02_VERSION == "LA-A02-0.1.0", "Pull the latest repository and restart the kernel"
DATA_ROOT = binance_root(REPO)
OUT = REPO / "notebooks" / "Liquidity_Absorption" / "02"
CFG = Config()
print("A02:", LA_A02_VERSION, " | Binance:", DATA_ROOT, " | Outputs:", OUT)
print("Candidate dates:", CFG.start, "to", CFG.end)
print("Reminder: EXPLORATORY FEATURE GENERATION; NO P&L")

In [ ]:
result = run_a02(DATA_ROOT, OUT, cfg=CFG)
result

In [ ]:
display(pd.read_csv(OUT / "02_coverage.csv"))
levels = pd.read_csv(OUT / "02_price_only_candidates.csv")
features = pd.read_csv(OUT / "02_causal_flow_features.csv")
print("Price-defined candidates:", len(levels), " | Enriched rows:", len(features))
assert len(levels) == len(features), "Candidate/enrichment mismatch"
assert not ({"delta_5m", "delta_15m", "cvd_utc_session", "pnl", "future_return"} & set(levels.columns)), "Price-only file contains nonprice features"
if len(levels):
    display(levels.groupby(["symbol", "side", "level_type"]).size().rename("candidates").reset_index())
    display(levels.head(10))
if len(features):
    display(features.groupby(["symbol", "profile_status"]).size().rename("events").reset_index())
    display(features[["symbol", "side", "signal_time_utc", "level_price", "delta_5m", "delta_15m", "zone_quote_last5m", "zone_quote_prev5m", "near_zone_share_last5m", "profile_status"]].head(10))

## Review before Stage A03

- Verify event timestamps use the **one-minute candle close** and reference levels were known at or before that time.
- Check how many events each source produces, including duplicates/overlapping levels; do not silently combine them.
- Inspect `profile_status`. `MISSING_TRADE_DAY` / `MISSING_TRADE_MINUTE` mean **no usable profile observation**, not zero volume. `PRESENT` means the preceding 10 minutes were available; no economic edge is inferred.
- Level proximity uses a **2bp placeholder** for crypto, not your MNQ 2–4 tick observation. The price-only levels are pragmatic proxies, not a faithful reconstruction of discretionary volume-profile zones.
- No fixed stop, +10 MNQ point break-even, structural trailing stops, profit targets or executable fills are modeled here. Those belong in Stage A03 once this feature contract is reviewed.

Commit/push this executed notebook and the CSV/JSON in `notebooks/Liquidity_Absorption/02/`. A full 92-day trade scan is substantial but uses Parquet batches to limit working memory. **Do not use any future-return or P&L information to retroactively tune the event definitions.**